<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/LTX_Director2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LTX-2.3 完整版：支持官方全部 12 个示例工作流

⚠️ **磁盘/显存需求提示**
- 官方 12 个 JSON 统一使用 46GB 的 `ltx-2.3-22b-dev.safetensors` 完整检查点（音频 VAE、文本投影均内置其中）
- 基础套装约 **67GB 磁盘**（检查点 46 + Gemma fp8 13 + 蒸馏 LoRA 7 + 上采样器），每个 IC-LoRA 另加 1~3GB
- 建议使用 **A100 / L4 大磁盘运行时**；免费 T4 磁盘和显存都很勉强
- 不需要的工作流请在 Cell 2 配置区把对应开关设为 `False`


In [ ]:
# ==========================================
# Cell 1: ComfyUI + 官方 12 工作流所需全部节点
# ==========================================
import os, sys, subprocess
from pathlib import Path

print("=== 🚀 安装 ComfyUI + 节点包 ===")

BASE_DIR = Path("/content")
COMFY_DIR = BASE_DIR / "ComfyUI"
CUSTOM_NODES_DIR = COMFY_DIR / "custom_nodes"

def run(cmd, cwd=None, check=False):
    print(" ".join(map(str, cmd)))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-2000:])
    if r.returncode != 0:
        print(r.stderr.strip()[-2000:])
        if check:
            raise RuntimeError("命令失败: " + " ".join(map(str, cmd)))
    return r

os.chdir(BASE_DIR)

# ComfyUI 本体
if not COMFY_DIR.exists():
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR)], check=True)
else:
    run(["git", "pull"], cwd=COMFY_DIR)

os.chdir(COMFY_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer", "opencv-python-headless"], check=True)

CUSTOM_NODES_DIR.mkdir(parents=True, exist_ok=True)

def install_node(repo_url):
    name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
    path = CUSTOM_NODES_DIR / name
    if not path.exists():
        run(["git", "clone", repo_url, str(path)], check=True)
    else:
        run(["git", "pull"], cwd=path)
    req = path / "requirements.txt"
    if req.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

# ==========================================
# ✅ 官方 12 个 2.3 示例工作流所需节点（已逐一核对 JSON）
# ==========================================
required_nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/Lightricks/ComfyUI-LTXVideo.git",       # LTXV* / IC-LoRA / Guider 等核心节点
    "https://github.com/kijai/ComfyUI-KJNodes.git",              # GetImageSizeAndCount / ImagePadForOutpaintTargetSize
    "https://github.com/ClownsharkBatwing/RES4LYF.git",          # ClownSampler_Beta（T2V/I2V 基础工作流也用）
    "https://github.com/cubiq/ComfyUI_essentials.git",           # SimpleMath+（HDR / MotionTrack / UnionControl）
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",  # DWPreprocessor / CannyEdgePreprocessor（UnionControl）
    "https://github.com/yuvraj108c/ComfyUI-Video-Depth-Anything.git",  # 深度提取（UnionControl）
    "https://github.com/WhatDreamsCost/WhatDreamsCost-ComfyUI.git",    # 原 Director Hotfix 工作流
    "https://github.com/rgthree/rgthree-comfy.git",              # 可选 UI 增强
]

for repo in required_nodes:
    install_node(repo)

# 可选加速，失败不影响
run([sys.executable, "-m", "pip", "install", "-q", "sageattention"])

# ✅ 验证 sageattention 是否真正可用（安装成功 != 可用）
SAGE_FLAG = BASE_DIR / ".sage_ok"
try:
    import importlib
    importlib.import_module("sageattention")
    SAGE_FLAG.write_text("1")
    print("⚡ sageattention 可用，Cell 4 启动时将启用 Sage Attention 加速")
except Exception as e:
    SAGE_FLAG.write_text("0")
    print(f"⚠️ sageattention 不可用，将回退到默认注意力（不影响运行）: {e}")

print("\n✅ Cell 1 完成")


In [ ]:
# ==========================================
# Cell 2: 下载官方 12 个工作流所需模型（按开关选择）
# ==========================================
import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

COMFY_DIR = Path("/content/ComfyUI")
MODELS = COMFY_DIR / "models"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("✅ 已读取 HF_TOKEN")
except Exception:
    pass

# ==========================================
# 🔧 配置区：按需开关，省磁盘
# ==========================================
DOWNLOAD_OFFICIAL_BASE = True    # 基础三件套：46GB dev 检查点 + Gemma + 蒸馏 LoRA + 空间上采样器
                                 # ✅ 开启后 T2V / I2V / T2A 三个基础工作流开箱即用
GEMMA_VARIANT = "fp8"            # "full"(~23GB 原版) / "fp8"(~13GB 推荐) / "fp4"(~8GB 仅新架构 GPU)

# ---- IC-LoRA 工作流开关 ----
WF_UNION_CONTROL    = True   # 深度+姿态+边缘控制（另需深度模型，DWPose 首次运行自动下载）
WF_MOTION_TRACK     = True   # 运动轨迹控制
WF_INPAINT_OUTPAINT = True   # 局部重绘 + 画幅扩展（共用一个 LoRA）
WF_INGREDIENTS      = True   # 多参考图"配料"合成
WF_PIXEL_UPSCALER   = True   # 生成式超分（x4 + x2）
WF_LIPDUB           = True   # 口型配音
WF_HDR              = False  # HDR 输出（专业调色用）
WF_V2V_SHAVE        = False  # V2V 示例（instant-shave 演示 LoRA）

DOWNLOAD_KIJAI_FP8_STACK = False  # 原 Director Hotfix 工作流的轻量 fp8 模型栈（与官方 JSON 不通用）
COPY_WORKFLOW_JSONS = True        # 把 12 个官方 JSON 复制到 ComfyUI 工作流目录

# ==========================================
# 下载任务表
# ==========================================
# 注意：官方 JSON 里 LoRA 路径写死为 loras/ltxv/ltx2/ 子目录，必须放对位置
LORA_DIR = MODELS / "loras" / "ltxv" / "ltx2"

tasks = []  # (repo_id, filename_in_repo, target_dir, target_name)

if DOWNLOAD_OFFICIAL_BASE:
    # 46GB 完整检查点：CheckpointLoaderSimple / LTXVAudioVAELoader / LTXAVTextEncoderLoader 三处都指向它
    tasks.append(("Lightricks/LTX-2.3", "ltx-2.3-22b-dev.safetensors",
                  MODELS / "checkpoints", None))
    # 蒸馏 LoRA（所有 12 个工作流都挂载它实现少步快速生成）
    tasks.append(("Lightricks/LTX-2.3", "ltx-2.3-22b-distilled-lora-384-1.1.safetensors",
                  LORA_DIR, None))
    # 两阶段工作流的潜空间空间上采样器
    tasks.append(("Lightricks/LTX-2.3", "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
                  MODELS / "latent_upscale_models", None))
    # Gemma 3 文本编码器
    gemma_files = {
        "full": "split_files/text_encoders/gemma_3_12B_it.safetensors",
        "fp8":  "split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors",
        "fp4":  "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors",
    }
    tasks.append(("Comfy-Org/ltx-2", gemma_files[GEMMA_VARIANT],
                  MODELS / "text_encoders", None))

if WF_UNION_CONTROL:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control",
                  "ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors", LORA_DIR, None))
if WF_MOTION_TRACK:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-Motion-Track-Control",
                  "ltx-2.3-22b-ic-lora-motion-track-control-ref0.5.safetensors", LORA_DIR, None))
if WF_INPAINT_OUTPAINT:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-In-Outpainting",
                  "ltx-2.3-22b-ic-lora-in-outpainting-0.9.safetensors", LORA_DIR, None))
if WF_INGREDIENTS:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-Ingredients",
                  "ltx-2.3-22b-ic-lora-ingredients-0.9.safetensors", LORA_DIR, None))
if WF_PIXEL_UPSCALER:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-Pixel-Spatial-Upscaler",
                  "ltx-2.3-22b-ic-lora-pixel-spatial-upscaler-x4-0.9.safetensors", LORA_DIR, None))
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-Pixel-Spatial-Upscaler",
                  "ltx-2.3-22b-ic-lora-pixel-spatial-upscaler-x2-0.9.safetensors", LORA_DIR, None))
if WF_LIPDUB:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-LipDub",
                  "ltx-2.3-22b-ic-lora-lipdub-0.9.safetensors", LORA_DIR, None))
if WF_HDR:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-HDR",
                  "ltx-2.3-22b-ic-lora-hdr-0.9.safetensors", LORA_DIR, None))
if WF_V2V_SHAVE:
    tasks.append(("Lightricks/LTX-2.3-22b-IC-LoRA-Instant-Shave",
                  "ltx-2.3-22b-ic-lora-instant-shave-0.9.safetensors", LORA_DIR, None))

if DOWNLOAD_KIJAI_FP8_STACK:
    tasks += [
        ("Kijai/LTX2.3_comfy", "diffusion_models/ltx-2.3-22b-distilled-1.1_transformer_only_fp8_scaled.safetensors", MODELS / "unet", None),
        ("Kijai/LTX2.3_comfy", "text_encoders/ltx-2.3_text_projection_bf16.safetensors", MODELS / "text_encoders", None),
        ("Kijai/LTX2.3_comfy", "vae/taeltx2_3.safetensors", MODELS / "vae", None),
        ("Kijai/LTX2.3_comfy", "vae/LTX23_video_vae_bf16.safetensors", MODELS / "vae", None),
        ("Kijai/LTX2.3_comfy", "vae/LTX23_audio_vae_bf16.safetensors", MODELS / "vae", None),
    ]

def download(repo_id, filename, target_dir, target_name=None):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    name = target_name or Path(filename).name
    final = target_dir / name
    if final.exists():
        print(f"⏭️  已存在: {final.name}")
        return
    print(f"⬇️  {repo_id} :: {filename}")
    cached = hf_hub_download(repo_id=repo_id, filename=filename)
    shutil.copy2(cached, final)
    print(f"✅ {final}")

for t in tasks:
    try:
        download(*t)
    except Exception as e:
        print(f"❌ 失败 {t[1]}: {e}")

# Gemma 变体重命名兼容：官方 JSON 默认写的是 comfy_gemma_3_12B_it.safetensors
# 建立同名软链接，加载工作流时无需手动改下拉框
if DOWNLOAD_OFFICIAL_BASE:
    te_dir = MODELS / "text_encoders"
    gemma_real = te_dir / Path(gemma_files[GEMMA_VARIANT]).name
    gemma_alias = te_dir / "comfy_gemma_3_12B_it.safetensors"
    if gemma_real.exists() and not gemma_alias.exists():
        os.symlink(gemma_real, gemma_alias)
        print(f"🔗 已创建别名: {gemma_alias.name} -> {gemma_real.name}")

# Union Control 的视频深度模型（不预下载也行，节点首次运行会自动拉取）
if WF_UNION_CONTROL:
    try:
        snapshot_download(repo_id="depth-anything/Video-Depth-Anything-Small",
                          allow_patterns=["*video_depth_anything_vits.pth*"],
                          local_dir=str(MODELS / "videodepthanything"))
        print("✅ Video-Depth-Anything-Small 就绪")
    except Exception as e:
        print(f"⚠️ 深度模型预下载失败（首次运行时节点会自动重试）: {e}")

# 把官方 12 个工作流 JSON 复制到 ComfyUI 工作流目录，UI 里直接打开
if COPY_WORKFLOW_JSONS:
    src = COMFY_DIR / "custom_nodes" / "ComfyUI-LTXVideo" / "example_workflows" / "2.3"
    dst = COMFY_DIR / "user" / "default" / "workflows" / "LTX-2.3-official"
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        n = 0
        for f in src.glob("*.json"):
            shutil.copy2(f, dst / f.name)
            n += 1
        print(f"✅ 已复制 {n} 个官方工作流到 {dst}")

print("\n✅ Cell 2 完成")
print("💡 DWPose 预处理模型（dw-ll_ucoco_384）由 controlnet_aux 首次使用时自动下载")


In [ ]:
# ==========================================
# Cell 3: FRP 内网穿透，可选
# ==========================================
import os, subprocess
from pathlib import Path

print("=== 🌐 配置 FRP，可选 ===")

try:
    from google.colab import userdata
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")
except Exception:
    VPS_IP = None
    FRP_TOKEN = None

if not VPS_IP or not FRP_TOKEN:
    print("⚠️ 没有 VPS_IP / FRP_TOKEN，跳过 FRP")
else:
    FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

    if not FRP_DIR.exists():
        subprocess.run(
            "wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content",
            shell=True,
            check=True
        )

    conf = f'''
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
'''
    (FRP_DIR / "frpc.toml").write_text(conf.strip(), encoding="utf-8")
    print("✅ FRP 配置完成")

In [ ]:
# ==========================================
# Cell 4: 启动 ComfyUI
# ==========================================
import os, subprocess, threading, time, configparser
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] Colab 保活中...")

threading.Thread(target=keep_alive, daemon=True).start()

# 启动 FRP
frpc = FRP_DIR / "frpc"
frpc_conf = FRP_DIR / "frpc.toml"

if frpc.exists() and frpc_conf.exists():
    def start_frpc():
        subprocess.run([str(frpc), "-c", str(frpc_conf)])

    threading.Thread(target=start_frpc, daemon=True).start()
    print("✅ FRP 已启动")
    print("👉 访问: http://cjp.usdream.dpdns.org:8090")
else:
    print("⚠️ 未启用 FRP")

# Manager private 模式，减少启动 fetch
manager_paths = [
    COMFY_DIR / "user" / "__manager" / "config.ini",
    COMFY_DIR / "user" / "default" / "ComfyUI-Manager" / "config.ini",
    COMFY_DIR / "custom_nodes" / "ComfyUI-Manager" / "config.ini",
]

for p in manager_paths:
    p.parent.mkdir(parents=True, exist_ok=True)
    cfg = configparser.ConfigParser()
    if p.exists():
        cfg.read(p)

    if "default" not in cfg:
        cfg["default"] = {}

    cfg["default"]["network_mode"] = "private"

    with open(p, "w") as f:
        cfg.write(f)

print("✅ Manager private 模式完成")

# ==========================================
# ✅ 启动参数：若 Cell 1 验证 sageattention 可用则启用加速
# ==========================================
launch_cmd = ["python", "main.py", "--dont-print-server"]

SAGE_FLAG = Path("/content/.sage_ok")
if SAGE_FLAG.exists() and SAGE_FLAG.read_text().strip() == "1":
    launch_cmd.append("--use-sage-attention")
    print("⚡ 已启用 Sage Attention 加速")
else:
    print("ℹ️ 未启用 Sage Attention，使用默认注意力")

os.chdir(COMFY_DIR)
print("🚀 启动 ComfyUI...")
subprocess.run(launch_cmd)